# Attendance Dataset Cleaning

Clean and validate the messy university attendance dataset.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## Load the Dataset

In [ ]:
df = pd.read_csv("Attendance_Messy_31.csv")
print("Loaded shape:", df.shape)

Loaded shape: (31, 5)


## Strip Whitespace from All Columns

In [3]:
str_cols = df.select_dtypes(include=["object"]).columns.tolist()
df["StudentID"] = df["StudentID"].astype("string").str.strip()
df["StudentID"] = df["StudentID"].replace("nan", np.nan)

C:\Windows\Temp\ipykernel_18016\3977891527.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include=["object"]).columns.tolist()


## Standardize Missing Values

In [4]:
missing_tokens = {"na", "n/a", "null", "-", "", "unknown", "not available", "0000-00-00"}
for col in str_cols:
    df[col] = df[col].apply(
        lambda x: np.nan if isinstance(x, str) and x.strip().lower() in missing_tokens else x
    )

## Inspect the Cleaned Data

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   StudentID          31 non-null     str    
 1   StudentName        31 non-null     str    
 2   Program            31 non-null     str    
 3   AttendancePercent  29 non-null     float64
 4   AttendanceStatus   31 non-null     str    
dtypes: float64(1), str(4)
memory usage: 1.3 KB


## Standardize Attendance Status

In [6]:
df["AttendanceStatus"] = (
    df["AttendanceStatus"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({"absent": "Absent", "ABSENT": "Absent",
          "present": "Present", "PRESENT": "Present"})
)

## Validate Attendance Percentage

In [7]:
df["AttendancePercent"] = pd.to_numeric(
    df["AttendancePercent"], errors="coerce"
)

df.loc[
    ~df["AttendancePercent"].between(0, 100),
    "AttendancePercent"
] = np.nan

## Fill Missing Attendance Percentages

In [8]:
median_attendance = df["AttendancePercent"].median()

df.loc[df["AttendancePercent"].isna(), "AttendancePercent"] = median_attendance

## Inspect a Sample Record

In [9]:
df.iloc[5]

StudentID                 STU406
StudentName              Sai Rao
Program              Electronics
AttendancePercent           73.0
AttendanceStatus         Present
Name: 5, dtype: object

## Validate Attendance Data

In [10]:
print(df.isna().sum())
print("Duplicate IDs:", df["StudentID"].duplicated().sum())
print(df["AttendanceStatus"].value_counts(dropna=False))
print(df["AttendancePercent"].describe())

StudentID            0
StudentName          0
Program              0
AttendancePercent    0
AttendanceStatus     0
dtype: int64
Duplicate IDs: 0
AttendanceStatus
Present    16
Absent     15
Name: count, dtype: int64
count    31.000000
mean     73.806452
std      11.522209
min      55.000000
25%      64.500000
50%      73.000000
75%      81.500000
max      99.000000
Name: AttendancePercent, dtype: float64


## Reload and Verify the Saved Dataset

In [ ]:
output_path = "Attendance_data_cleaned_31.csv"
df.to_csv(output_path, index=False)

In [12]:
att = pd.read_csv(output_path)
print(att.head())
print(att.shape)

  StudentID     StudentName           Program  AttendancePercent AttendanceStatus
0    STU401  Aarav Malhotra      Data Science               73.0          Present
1    STU402    Vivaan Gupta       Electronics               73.0           Absent
2    STU403      Aditya Das  Computer Science               86.0          Present
3    STU404    Vihaan Menon       Electronics               59.0           Absent
4    STU405    Arjun Pillai        Mechanical               76.0          Present
(31, 5)
